# Embedding the VaR backtest report in a notebook

`report.html` is script-driven, so it **won't** render if you paste it into a
Markdown cell &mdash; Jupyter strips `<script>` tags (and `display(HTML(...))`
is sanitized the same way in JupyterLab). The reliable trick is an **iframe**,
which runs its own JavaScript in a separate document.

The cell below bakes the results CSV into the page, writes
`report_embedded.html`, and shows it via `IFrame`. The report then renders
fully interactive (metric toggles, hover read-out, dotted "current" line).

**Usage**
- Keep this notebook in the same folder as `report.html` and
  `var_backtest_results.csv` (the repo root), **or** pass explicit paths to
  `show_report(...)`.
- Re-run the cell after regenerating the CSV to refresh the charts.
- If the iframe shows blank, the files probably aren't on the path Jupyter is
  serving &mdash; check the working directory with `import os; os.getcwd()`.

In [ ]:
# Display the standalone VaR report inline. Works in classic Notebook and JupyterLab.
import base64
from pathlib import Path
from IPython.display import IFrame

def show_report(html_path="report.html",
                csv_path="var_backtest_results.csv",
                out="report_embedded.html",
                height=900):
    """Bake the CSV into report.html and embed it as an interactive iframe.

    An iframe runs its own scripts (unlike Markdown cells or display(HTML),
    which Jupyter sanitizes), so the charts, toggles and hover all work.
    """
    html_path, csv_path = Path(html_path), Path(csv_path)
    for p in (html_path, csv_path):
        if not p.exists():
            raise FileNotFoundError(
                f"{p} not found - run from the repo folder or pass an explicit path."
            )
    page = html_path.read_text(encoding="utf-8")
    b64 = base64.b64encode(csv_path.read_bytes()).decode()
    # Auto-load the data on page load so it renders without the file picker.
    page = page.replace(
        "</body>",
        f'<script>window.addEventListener("load",()=>loadText(atob("{b64}")));</script></body>',
    )
    Path(out).write_text(page, encoding="utf-8")
    return IFrame(out, width="100%", height=height)

show_report()